# Fish disease classifier v2 - robustness without new field photos

**Before you start:** Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all** (about 25-40 min).

**What is new compared with v1**
* **Stronger training "damage"** (rotation, random crops/aspect ratios, low resolution, blur, JPEG artifacts, noise) so the model
  copes with phone/web/screenshot photos instead of only the original 4000x3000 farm photos.
* **A second, differently-styled source** (your old Kaggle 7-class set, cleaned) so the model also sees web-style photos. Its extra
  diseases (aeromoniasis, saprolegniasis, parasitic, white tail) are merged into ONE class `other_disease`, so unfamiliar sick fish
  have a "sick, not identified" answer instead of falling into "healthy".
* **An honest external test:** the Kaptai/haor photos (never trained on; near-copies removed) measure the failure you saw:
  *sick fish called healthy*.
* **It compares 3 recipes and picks the winner by itself**, then trains that one on everything and exports it.

**You get one download:** `fish_model_bundle_v2.zip` = `fish_disease_model.tflite` + `labels.txt` + `report_v2.txt`.

In [ ]:
# 1. Setup
import os, glob, re, subprocess, shutil, hashlib
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from PIL import Image, ImageOps
import tensorflow as tf
from sklearn.model_selection import StratifiedGroupKFold

SMOKE = os.environ.get("FISH_SMOKE") == "1"          # tiny offline self-test only; leave unset in Colab
WORK = os.environ.get("FISH_WORK", "/content")
MATSYA_URL = "https://data.mendeley.com/public-api/zip/sxkynv9t7n/download/2"
RAW, EXTRACTED, SMALL, OUT = f"{WORK}/matsya", f"{WORK}/matsya_x", f"{WORK}/small", f"{WORK}/out"

IMG = 128 if SMOKE else 224        # model input size - must match the Flutter app (it reads it from the model)
MAX_PER_FISH = 30
N_FOLDS = 2 if SMOKE else 5        # only used to carve out one held-out group of Matsya fish
HEAD_EPOCHS = 1 if SMOKE else 6
FT_EPOCHS = 1 if SMOKE else 14
BATCH, SEED = 32, 42
PRETRAINED = not SMOKE
CLASSES = ["bacterial_gill_disease", "bacterial_red_disease", "eus", "other_disease", "healthy"]
H = CLASSES.index("healthy")
MATSYA_MAP = {"Bacterial Gill Diseases": "bacterial_gill_disease", "Bacterial Red Diseases": "bacterial_red_disease",
              "Epizootic Ulcerative Syndrome (EUS)": "eus", "Healthy Fish": "healthy"}
tf.keras.utils.set_random_seed(SEED)
os.makedirs(OUT, exist_ok=True)
LOG = []
def log(*a):
    s = " ".join(str(x) for x in a); print(s); LOG.append(s)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "NONE -> set Runtime > T4 GPU or this is very slow")

TensorFlow 2.21.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# 2. Get the three datasets (each step is skipped if already downloaded)
if SMOKE:
    WEB7_DIR, KAPTAI_DIR = os.environ["FISH_WEB7"], os.environ["FISH_KAPTAI"]
else:
    if not os.path.exists(EXTRACTED):
        if not glob.glob(f"{RAW}/*/MatsyaDx-BD/metadata.csv"):
            subprocess.run(["wget", "-q", "-L", "-O", f"{WORK}/matsya.zip", MATSYA_URL], check=True)
            subprocess.run(["unzip", "-q", "-o", f"{WORK}/matsya.zip", "-d", RAW], check=True)
            os.remove(f"{WORK}/matsya.zip")
        subprocess.run("apt-get -qq install -y p7zip-full", shell=True)
        meta_dir = os.path.dirname(glob.glob(f"{RAW}/*/MatsyaDx-BD/metadata.csv")[0])
        os.makedirs(EXTRACTED, exist_ok=True)
        for a in sorted(glob.glob(meta_dir + "/*.7z")):
            print("extracting", os.path.basename(a), flush=True)
            subprocess.run(["7z", "x", "-y", a, f"-o{EXTRACTED}"], check=True, stdout=subprocess.DEVNULL)
    subprocess.run("pip install -q kagglehub", shell=True)
    import kagglehub
    WEB7_DIR = kagglehub.dataset_download("subirbiswas19/freshwater-fish-disease-aquaculture-in-south-asia")
    KAPTAI_DIR = kagglehub.dataset_download("utpolkantidas/fresh-water-fish-disease-dataset")
META = glob.glob(f"{RAW}/*/MatsyaDx-BD/metadata.csv")[0]
print("Matsya metadata:", META, "\nweb 7-class set:", WEB7_DIR, "\nKaptai/haor set:", KAPTAI_DIR)

Using Colab cache for faster access to the 'freshwater-fish-disease-aquaculture-in-south-asia' dataset.
Using Colab cache for faster access to the 'fresh-water-fish-disease-dataset' dataset.
Matsya metadata: /content/matsya/MatsyaDx-BD An image dataset of freshwater fish di/MatsyaDx-BD/metadata.csv 
web 7-class set: /kaggle/input/freshwater-fish-disease-aquaculture-in-south-asia 
Kaptai/haor set: /kaggle/input/fresh-water-fish-disease-dataset


In [ ]:
# 3. Helpers: image lists, perceptual hashes, near-duplicate clusters
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
def list_images(root):
    return sorted(os.path.join(r, f) for r, _, fs in os.walk(root) for f in fs if f.lower().endswith(IMG_EXT))
def md5(p): return hashlib.md5(open(p, "rb").read()).hexdigest()
def dhash(path):
    im = Image.open(path).convert("RGB")
    def h(x):
        g = np.asarray(x.convert("L").resize((9, 8), Image.LANCZOS), dtype=np.int16); return (g[:, 1:] > g[:, :-1]).flatten()
    return h(im), h(ImageOps.mirror(im))
def min_dist(Ha, Hb):            # smallest Hamming distance between each row of Ha and any row of Hb (mirror-aware)
    out = np.full(len(Ha), 64)
    for i in range(0, len(Ha), 256):
        blk = Ha[i:i + 256]
        d = np.minimum((blk[:, None, :] != Hb[0][None]).sum(-1), (blk[:, None, :] != Hb[1][None]).sum(-1))
        out[i:i + 256] = d.min(1)
    return out
def clusters(hashes, flips, th=4):
    n = len(hashes); parent = list(range(n))
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for i in range(0, n, 256):
        blk = slice(i, min(i + 256, n))
        m = np.minimum((hashes[blk, None, :] != hashes[None]).sum(-1), (hashes[blk, None, :] != flips[None]).sum(-1))
        for a, b in np.argwhere(m <= th):
            if i + a != b: parent[find(b)] = find(i + a)
    return np.array([find(i) for i in range(n)])

In [ ]:
# 4. EXTERNAL TEST SET (Kaptai/haor photos): never trained on. Photos filed under 2+ labels are dropped.
KT_RULES = [("healthy", "healthy"), (r"eus|redspot", "eusred"), ("gill", "gill")]      # anything else = sick, not covered by our classes
rows = []
for p in list_images(KAPTAI_DIR):
    folder = os.path.basename(os.path.dirname(p)); cat = "othersick"
    for pat, c in KT_RULES:
        if re.search(pat, folder.lower()): cat = c; break
    rows.append(dict(path=p, folder=folder, cat=cat, md5=md5(p)))
kt = pd.DataFrame(rows)
conflict = kt.groupby("md5")["folder"].nunique(); n_all = len(kt)
kt = kt[kt["md5"].map(conflict) == 1].drop_duplicates("md5").reset_index(drop=True)
kt["y"] = 0                      # placeholder: the input pipeline expects a label column; test labels are read from "cat"
log(f"Kaptai/haor test set: {n_all} files -> {len(kt)} clean photos (duplicates and photos filed under 2+ labels removed)")
log(kt["cat"].value_counts().to_string())
kt_h = [dhash(p) for p in kt["path"]]; KTH = (np.array([a for a, _ in kt_h]), np.array([b for _, b in kt_h]))

Kaptai/haor test set: 133 files -> 103 clean photos (duplicates and photos filed under 2+ labels removed)
cat
eusred       50
healthy      28
othersick    19
gill          6


In [ ]:
# 5. SOURCE A - MatsyaDx-BD (farm photos, grouped by individual fish)
dm = pd.read_csv(META)
dm["path"] = EXTRACTED + "/" + dm["image_path"]
dm["label"] = dm["health_condition"].map(MATSYA_MAP)
dm["y"] = dm["label"].map({c: i for i, c in enumerate(CLASSES)})
assert dm["y"].notna().all(), "unexpected class name in metadata.csv"
dm["group"] = "m|" + dm["health_condition"] + "|" + dm["fish_category"] + "|" + dm["specimen_id"]
assert dm["path"].map(os.path.exists).all(), "some Matsya files are missing - extraction incomplete?"
def pick_even(g, k):
    g = g.sort_values("image_id")
    return g if len(g) <= k else g.iloc[np.linspace(0, len(g) - 1, k).round().astype(int)]
dm = pd.concat([pick_even(g, MAX_PER_FISH) for _, g in dm.groupby("group")]).reset_index(drop=True)
dm["source"] = "matsya"
log(f"\nMatsyaDx-BD: {len(dm)} photos from {dm['group'].nunique()} fish\n" + dm.groupby("label").agg(photos=("path", "size"), fish=("group", "nunique")).to_string())


MatsyaDx-BD: 1511 photos from 86 fish
                        photos  fish
label                               
bacterial_gill_disease     275    27
bacterial_red_disease      634    35
eus                        302    14
healthy                    300    10


In [ ]:
# 6. SOURCE B - the old 7-class web set, cleaned (duplicates, near-copies, conflicting labels, copies of the test photos removed)
WEB_RULES = [("gill", "bacterial_gill_disease"), (r"\bred\b", "bacterial_red_disease"), ("healthy", "healthy"),
             ("aeromon", "other_disease"), ("saprolegn|fungal", "other_disease"), ("parasit", "other_disease"), (r"white.?tail|viral", "other_disease")]
rows, unmapped = [], set()
for p in list_images(WEB7_DIR):
    rel = os.path.relpath(os.path.dirname(p), WEB7_DIR).lower(); lab = None
    for pat, l in WEB_RULES:
        if re.search(pat, rel): lab = l; break
    if lab is None: unmapped.add(rel); continue
    rows.append(dict(path=p, folder=rel, label=lab, md5=md5(p)))
dw = pd.DataFrame(rows)
log("\nWeb 7-class set: folder -> class mapping used:\n" + dw.groupby(["folder", "label"]).size().to_string())
if unmapped: log("WARNING - folders ignored (no rule matched):", sorted(unmapped))
assert dw["label"].nunique() >= 4, "could not map the web dataset folders - check the mapping printed above"
n0 = len(dw)
bad = dw.groupby("md5")["label"].nunique(); dw = dw[dw["md5"].map(bad) == 1].drop_duplicates("md5").reset_index(drop=True)
hh = [dhash(p) for p in dw["path"]]; HA, HF = np.array([a for a, _ in hh]), np.array([b for _, b in hh])
near_test = min_dist(HA, KTH) <= 4                                    # near-copies of the external test photos
dw = dw[~near_test].reset_index(drop=True); HA, HF = HA[~near_test], HF[~near_test]
cid = clusters(HA, HF)
mixed = pd.Series(dw["label"].values).groupby(cid).nunique(); keep = ~np.isin(cid, mixed[mixed > 1].index)
dw, cid = dw[keep].reset_index(drop=True), cid[keep]
dw["group"] = ["w|" + str(c) for c in cid]; dw["y"] = dw["label"].map({c: i for i, c in enumerate(CLASSES)}); dw["source"] = "web"
log(f"web set: {n0} files -> {len(dw)} kept ({int(near_test.sum())} were near-copies of the Kaptai test photos, others exact/conflicting duplicates)")
log(dw.groupby("label").agg(photos=("path", "size"), independent_pictures=("group", "nunique")).to_string())


Web 7-class set: folder -> class mapping used:
folder                                                                                     label                 
freshwater fish disease aquaculture in south asia/test/bacterial diseases - aeromoniasis   other_disease             100
freshwater fish disease aquaculture in south asia/test/bacterial gill disease              bacterial_gill_disease    100
freshwater fish disease aquaculture in south asia/test/bacterial red disease               bacterial_red_disease     100
freshwater fish disease aquaculture in south asia/test/fungal diseases saprolegniasis      other_disease             100
freshwater fish disease aquaculture in south asia/test/healthy fish                        healthy                   100
freshwater fish disease aquaculture in south asia/test/parasitic diseases                  other_disease             100
freshwater fish disease aquaculture in south asia/test/viral diseases white tail disease   other_disease        

In [ ]:
# 7. Make uniform small JPEG copies of every image we will use
os.makedirs(SMALL, exist_ok=True)
def small_path(prefix, n): return [f"{SMALL}/{prefix}{i:05d}.jpg" for i in range(n)]
dm["small"], dw["small"], kt["small"] = small_path("m", len(dm)), small_path("w", len(dw)), small_path("k", len(kt))
def shrink(args):
    src, dst = args
    if os.path.exists(dst): return
    im = Image.open(src); im.draft("RGB", (1000, 1000))
    im = ImageOps.exif_transpose(im).convert("RGB"); im.thumbnail((448, 448), Image.LANCZOS); im.save(dst, quality=95)
with ThreadPoolExecutor(max_workers=os.cpu_count() or 2) as ex:
    for d in (dm, dw, kt): list(ex.map(shrink, zip(d["path"], d["small"])))
print("small copies ready:", len(dm), len(dw), len(kt))

small copies ready: 1511 1704 103


In [ ]:
# 8. Input pipeline (plain vs STRONG augmentation) + model
def load_plain(path, y):
    x = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    return tf.image.resize(x, (IMG, IMG), method="bilinear", antialias=True), y          # squash to square, same as the app

def load_strong(path, y):                              # random crop with random aspect ratio, THEN squash (portrait/zoomed photos)
    raw = tf.io.decode_jpeg(tf.io.read_file(path), channels=3); sh = tf.shape(raw)
    ch = tf.cast(tf.round(tf.cast(sh[0], tf.float32) * tf.random.uniform([], 0.65, 1.0)), tf.int32)
    cw = tf.cast(tf.round(tf.cast(sh[1], tf.float32) * tf.random.uniform([], 0.65, 1.0)), tf.int32)
    oy = tf.random.uniform([], 0, sh[0] - ch + 1, dtype=tf.int32); ox = tf.random.uniform([], 0, sh[1] - cw + 1, dtype=tf.int32)
    x = tf.image.crop_to_bounding_box(raw, oy, ox, ch, cw)
    return tf.image.resize(x, (IMG, IMG), method="bilinear", antialias=True), y

def augment_plain(x, y):                               # the v1 recipe
    x = tf.image.random_flip_left_right(x)
    size = tf.cast(tf.round(IMG * tf.random.uniform([], 1.0, 1.25)), tf.int32)
    x = tf.image.random_crop(tf.image.resize(x, tf.stack([size, size])), (IMG, IMG, 3))
    x = tf.image.random_brightness(x, 25.0); x = tf.image.random_contrast(x, 0.8, 1.2); x = tf.image.random_saturation(x, 0.8, 1.2)
    return tf.ensure_shape(tf.clip_by_value(x, 0.0, 255.0), (IMG, IMG, 3)), y

def augment_strong(x, y):
    x = tf.image.random_flip_left_right(x)
    ang, s = tf.random.uniform([], -0.35, 0.35), tf.random.uniform([], 0.9, 1.15)          # +-20 degrees, zoom, shift
    c, sn, mid = tf.cos(ang), tf.sin(ang), IMG / 2.0
    tx, ty = tf.random.uniform([], -0.06, 0.06) * IMG, tf.random.uniform([], -0.06, 0.06) * IMG
    a0, a1, b0, b1 = c / s, -sn / s, sn / s, c / s
    mat = tf.stack([a0, a1, mid - (a0 * mid + a1 * mid) + tx, b0, b1, mid - (b0 * mid + b1 * mid) + ty, 0.0, 0.0])[None]
    x = tf.raw_ops.ImageProjectiveTransformV3(images=x[None], transforms=mat, output_shape=[IMG, IMG],
                                              interpolation="BILINEAR", fill_mode="REFLECT", fill_value=0.0)[0]
    def lowres():                                       # look like a small, blurry web photo
        sz = tf.random.uniform([], 64, 300, dtype=tf.int32)
        return tf.image.resize(tf.image.resize(x, tf.stack([sz, sz]), antialias=True), (IMG, IMG))
    x = tf.cond(tf.random.uniform([]) < 0.6, lowres, lambda: tf.identity(x))
    x = tf.cond(tf.random.uniform([]) < 0.5,            # JPEG artifacts
                lambda: tf.image.random_jpeg_quality(tf.clip_by_value(x, 0.0, 255.0) / 255.0, 25, 95) * 255.0, lambda: tf.identity(x))
    x = x + tf.random.normal(tf.shape(x), stddev=tf.random.uniform([], 0.0, 5.0))
    x = tf.image.random_brightness(x, 30.0); x = tf.image.random_contrast(x, 0.75, 1.3); x = tf.image.random_saturation(x, 0.7, 1.3)
    x = tf.cond(tf.random.uniform([]) < 0.05, lambda: tf.image.grayscale_to_rgb(tf.image.rgb_to_grayscale(x)), lambda: tf.identity(x))
    return tf.ensure_shape(tf.clip_by_value(x, 0.0, 255.0), (IMG, IMG, 3)), y

def make_ds(d, train=False, strong=False):
    ds = tf.data.Dataset.from_tensor_slices((d["small"].values, d["y"].values.astype("int32")))
    if train: ds = ds.shuffle(len(d), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_strong if (train and strong) else load_plain, tf.data.AUTOTUNE)
    if train: ds = ds.map(augment_strong if strong else augment_plain, tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

def build_model():
    base = tf.keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet" if PRETRAINED else None)
    inp = tf.keras.Input((IMG, IMG, 3), name="input")                # raw pixel values 0..255
    x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1)(inp)         # MobileNetV2 wants [-1, 1]: done inside the model
    x = base(x, training=False)
    x = tf.keras.layers.Dropout(0.3)(tf.keras.layers.GlobalAveragePooling2D()(x))
    return tf.keras.Model(inp, tf.keras.layers.Dense(len(CLASSES), name="output")(x)), base      # logits

def compile_(m, lr):
    m.compile(tf.keras.optimizers.Adam(lr), tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])

def train_model(train_df, strong):
    counts = np.bincount(train_df["y"], minlength=len(CLASSES))
    cw = {i: len(train_df) / (len(CLASSES) * max(int(c), 1)) for i, c in enumerate(counts)}
    tr = make_ds(train_df, True, strong); model, base = build_model()
    base.trainable = False; compile_(model, 1e-3); model.fit(tr, epochs=HEAD_EPOCHS, class_weight=cw, verbose=2)
    base.trainable = True
    for l in base.layers[:-50]: l.trainable = False
    for l in base.layers:
        if isinstance(l, tf.keras.layers.BatchNormalization): l.trainable = False
    compile_(model, 1e-4); model.fit(tr, epochs=FT_EPOCHS, class_weight=cw, verbose=2)
    return model

def predict_probs(model, d):
    z = model.predict(make_ds(d), verbose=0); e = np.exp(z - z.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)

In [ ]:
# 9. Compare three recipes. Each is judged on (a) held-out Matsya fish and (b) the external Kaptai/haor photos.
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
tr_i, ho_i = next(sgkf.split(dm, dm["y"], dm["group"]))
m_train, m_hold = dm.iloc[tr_i], dm.iloc[ho_i]
assert not set(m_train["group"]) & set(m_hold["group"])
log(f"\nHeld-out Matsya fish for the in-domain check: {m_hold['group'].nunique()} fish / {len(m_hold)} photos")
Rk, Gk = [CLASSES.index("bacterial_red_disease"), CLASSES.index("eus")], CLASSES.index("bacterial_gill_disease")

def evaluate(model):
    ph, pk = predict_probs(model, m_hold).argmax(1), predict_probs(model, kt).argmax(1)
    yh, sick_h = m_hold["y"].values, m_hold["y"].values != H
    sick_k, cat = kt["cat"].values != "healthy", kt["cat"].values
    return {"holdout_acc": float((ph == yh).mean()),
            "holdout_sick_called_healthy": float((ph[sick_h] == H).mean()),
            "kaptai_sick_called_healthy": float((pk[sick_k] == H).mean()),
            "kaptai_healthy_called_healthy": float((pk[~sick_k] == H).mean()),
            "kaptai_eus_redspot_found": float(np.isin(pk[cat == "eusred"], Rk).mean()) if (cat == "eusred").any() else float("nan"),
            "kaptai_gillrot_found": float((pk[cat == "gill"] == Gk).mean()) if (cat == "gill").any() else float("nan")}

RECIPES = {"A0 (v1 recipe: Matsya only)": (m_train, False),
           "A  (Matsya only + strong augmentation)": (m_train, True),
           "B  (Matsya + web set + strong augmentation)": (pd.concat([m_train, dw]).reset_index(drop=True), True)}
results = {}
for name, (d, strong) in RECIPES.items():
    log(f"\n=== training recipe {name}: {len(d)} photos ===")
    mdl = train_model(d, strong); results[name] = evaluate(mdl); del mdl; tf.keras.backend.clear_session()
log("\n##### COMPARISON (lower 'sick called healthy' is better) #####")
log(pd.DataFrame(results).T.round(3).to_string())
log("(v1 reference on the Kaptai photos: 65% of sick photos called healthy)")


Held-out Matsya fish for the in-domain check: 18 fish / 366 photos

=== training recipe A0 (v1 recipe: Matsya only): 1145 photos ===
Epoch 1/6
36/36 - 22s - 608ms/step - accuracy: 0.5266 - loss: 0.9319
Epoch 2/6
36/36 - 7s - 182ms/step - accuracy: 0.7362 - loss: 0.4833
Epoch 3/6
36/36 - 10s - 275ms/step - accuracy: 0.8262 - loss: 0.3578
Epoch 4/6
36/36 - 5s - 149ms/step - accuracy: 0.8550 - loss: 0.2919
Epoch 5/6
36/36 - 11s - 318ms/step - accuracy: 0.8978 - loss: 0.2166
Epoch 6/6
36/36 - 10s - 285ms/step - accuracy: 0.9118 - loss: 0.1932
Epoch 1/14
36/36 - 25s - 687ms/step - accuracy: 0.8166 - loss: 0.3452
Epoch 2/14
36/36 - 6s - 177ms/step - accuracy: 0.9476 - loss: 0.1074
Epoch 3/14
36/36 - 7s - 184ms/step - accuracy: 0.9817 - loss: 0.0382
Epoch 4/14
36/36 - 5s - 145ms/step - accuracy: 0.9939 - loss: 0.0149
Epoch 5/14
36/36 - 7s - 182ms/step - accuracy: 0.9991 - loss: 0.0078
Epoch 6/14
36/36 - 9s - 262ms/step - accuracy: 0.9974 - loss: 0.0073
Epoch 7/14
36/36 - 6s - 166ms/step - ac

In [ ]:
# 10. Pick the winner automatically, train it on ALL data, export
best_in = max(r["holdout_acc"] for r in results.values())
# a recipe must (1) not lose in-domain accuracy and (2) still call at least half of the HEALTHY external photos healthy,
# otherwise "always say sick" would trivially win
ok = {k: r for k, r in results.items() if r["holdout_acc"] >= best_in - 0.05 and r["kaptai_healthy_called_healthy"] >= 0.5}
if not ok:
    log("WARNING: no recipe kept at least half of the healthy external photos healthy; falling back to the in-domain criterion only")
    ok = {k: r for k, r in results.items() if r["holdout_acc"] >= best_in - 0.05}
choice = min(ok, key=lambda k: (ok[k]["kaptai_sick_called_healthy"], -ok[k]["holdout_acc"]))
log(f"\nChosen recipe: {choice}\n  (lowest 'sick called healthy' on the external photos among recipes that keep in-domain accuracy and still recognise healthy fish)")
final_df, strong = (pd.concat([dm, dw]).reset_index(drop=True), True) if choice.startswith("B") else (dm, choice.startswith("A "))
log(f"Final training on {len(final_df)} photos ({'with' if strong else 'without'} strong augmentation)")
final = train_model(final_df, strong)
try:
    tflite = tf.lite.TFLiteConverter.from_keras_model(final).convert()
except Exception as e:
    log("from_keras_model failed:", repr(e)[:200], "-> trying SavedModel route")
    final.export(f"{WORK}/saved_model"); tflite = tf.lite.TFLiteConverter.from_saved_model(f"{WORK}/saved_model").convert()
open(f"{OUT}/fish_disease_model.tflite", "wb").write(tflite)
open(f"{OUT}/labels.txt", "w").write("\n".join(CLASSES) + "\n")
try:
    it = tf.lite.Interpreter(model_content=tflite)
except Exception:
    subprocess.run("pip install -q ai-edge-litert", shell=True)
    from ai_edge_litert.interpreter import Interpreter
    it = Interpreter(model_content=tflite)
it.allocate_tensors(); i_in, i_out = it.get_input_details()[0], it.get_output_details()[0]
log("TFLite input:", [int(v) for v in i_in["shape"]], "| output:", [int(v) for v in i_out["shape"]])
sample = kt.sample(min(30, len(kt)), random_state=1)
keras_z = final.predict(make_ds(sample), verbose=0)
def tfl(x):
    it.set_tensor(i_in["index"], x.astype("float32")[None]); it.invoke(); return it.get_tensor(i_out["index"])[0].copy()
tfl_z = np.array([tfl(load_plain(p, 0)[0].numpy()) for p in sample["small"]])
log(f"TFLite vs Keras: max logit difference {np.abs(tfl_z - keras_z).max():.5f}, same answer on {(tfl_z.argmax(1) == keras_z.argmax(1)).mean():.0%} of photos")


Chosen recipe: B  (Matsya + web set + strong augmentation)
  (lowest 'sick called healthy' on the external photos among recipes that keep in-domain accuracy and still recognise healthy fish)
Final training on 3215 photos (with strong augmentation)
Epoch 1/6
101/101 - 50s - 491ms/step - accuracy: 0.4946 - loss: 1.3049
Epoch 2/6
101/101 - 37s - 364ms/step - accuracy: 0.6373 - loss: 0.8829
Epoch 3/6
101/101 - 35s - 348ms/step - accuracy: 0.6955 - loss: 0.7718
Epoch 4/6
101/101 - 44s - 436ms/step - accuracy: 0.7048 - loss: 0.6953
Epoch 5/6
101/101 - 74s - 736ms/step - accuracy: 0.7350 - loss: 0.6504
Epoch 6/6
101/101 - 37s - 363ms/step - accuracy: 0.7431 - loss: 0.6351
Epoch 1/14
101/101 - 55s - 547ms/step - accuracy: 0.6986 - loss: 0.7495
Epoch 2/14
101/101 - 37s - 366ms/step - accuracy: 0.7869 - loss: 0.5002
Epoch 3/14
101/101 - 35s - 351ms/step - accuracy: 0.8411 - loss: 0.3751
Epoch 4/14
101/101 - 37s - 366ms/step - accuracy: 0.8684 - loss: 0.3279
Epoch 5/14
101/101 - 40s - 395ms/step

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


TFLite input: [1, 224, 224, 3] | output: [1, 5]
TFLite vs Keras: max logit difference 0.00003, same answer on 100% of photos


In [ ]:
# 11. Resize sensitivity + one downloadable zip
def full_photo_input(path, method):
    im = Image.open(path); im.draft("RGB", (2000, 2000)); im = ImageOps.exif_transpose(im).convert("RGB")
    im.thumbnail((1024, 1024), Image.BILINEAR); return np.asarray(im.resize((IMG, IMG), method), dtype=np.float32)
rs = dm.sample(min(20, len(dm)), random_state=2)
ref = np.array([tfl(full_photo_input(p, Image.BILINEAR)).argmax() for p in rs["path"]])
for nm, meth in [("nearest-neighbour (the OLD app resize)", Image.NEAREST), ("box/average", Image.BOX)]:
    got = np.array([tfl(full_photo_input(p, meth)).argmax() for p in rs["path"]])
    log(f"{nm}: same answer as the training-style resize on {(got == ref).mean():.0%} of {len(rs)} full-size photos")
open(f"{OUT}/report_v2.txt", "w").write("\n".join(LOG))
shutil.make_archive(f"{WORK}/fish_model_bundle_v2", "zip", OUT)
print("model size: %.1f MB" % (os.path.getsize(f"{OUT}/fish_disease_model.tflite") / 1e6))
try:
    from google.colab import files
    files.download(f"{WORK}/fish_model_bundle_v2.zip")
except ImportError:
    print("saved:", f"{WORK}/fish_model_bundle_v2.zip")

nearest-neighbour (the OLD app resize): same answer as the training-style resize on 100% of 20 full-size photos
box/average: same answer as the training-style resize on 100% of 20 full-size photos
model size: 8.9 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### After the download
Paste the **COMPARISON** table and the "Chosen recipe" lines from `report_v2.txt` back into the chat.
The app then needs: the new `labels.txt` (5 classes, including `other_disease`) and wording for "Other disease signs".